# 06 · Classification, ROC-AUC, PR-AUC, and thresholds

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Train logistic regression; distinguish ranking from decisions; choose thresholds using validation data.


## Step 1 — Create an imbalanced problem

The positive class is less common, so accuracy alone can hide missed positives.


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5,
                           weights=[0.75, 0.25], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

X_fit, X_val, y_fit, y_val = train_test_split(X_train,y_train,stratify=y_train,random_state=7,test_size=0.25)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
model.fit(X_fit, y_fit)
print('Positive rate:', y.mean())


**Check your result:** Roughly one quarter of rows are positive.


## Step 2 — Choose a decision rule

Choose the threshold on validation data. Using test labels for this choice would bias the final score.


In [ ]:
thresholds = np.linspace(0.1,0.9,17)
val_prob = model.predict_proba(X_val)[:,1]
scores = [f1_score(y_val,val_prob >= t) for t in thresholds]
threshold = float(thresholds[np.argmax(scores)])
print('Validation threshold:', threshold, 'F1:', max(scores))


**Check your result:** The chosen threshold need not equal 0.5.


## Step 3 — Evaluate once

ROC-AUC measures ranking over all thresholds. Average precision summarizes the precision-recall curve and is useful under imbalance.


In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score
prob = model.predict_proba(X_test)[:,1]
pred = prob >= threshold
print({'roc_auc':roc_auc_score(y_test,prob),'average_precision':average_precision_score(y_test,prob),
       'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred), 'f1':f1_score(y_test,pred)})
print('Rows=true, columns=predicted:', confusion_matrix(y_test,pred))
assert roc_auc_score(y_test,prob) > 0.7


**Check your result:** Ranking beats random; the confusion matrix shows the decision tradeoff.


## Practice

Choose a validation threshold with recall at least 0.9, maximizing precision among eligible thresholds.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Calculate precision and recall for each candidate using only y_val. If none qualifies, report that the requirement cannot be met with the current candidates.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
